### Load the data from the CSV file.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

dirty_cafe_df = spark.read.format("csv")\
                .option("header","true")\
                .option("inferschema", "true")\
                .load("/Workspace/Users/raj.s@brilworks.com/data-engineering-learnings/exercises/session-4/dirty_cafe_sales.csv")

dirty_cafe_df.show()

### Total number of records in the data.

In [0]:
dirty_cafe_df.count()

#### Schema of the data.

In [0]:
dirty_cafe_df.printSchema()


### Transaction ID - Data Quality Check

In [0]:
# Are there any records where transaction IDs do not exist?

print("Are there any records where transaction IDs do not exist?")
dirty_cafe_df.filter(F.col("Transaction ID").isNull()).select("*").show()

# Are there any records where transaction IDs are not unique?

print("Are there any records where transaction IDs are not unique?")
dirty_cafe_df.groupBy("Transaction ID").count().filter(F.col("count") > 1).select("Transaction ID").show()

# Are there any Transaction IDs that do not follow the pattern of 'TXN_(\d{7})'?
print(r"Are there any Transaction IDs that do not follow the pattern of 'TXN_(\d{7})'?")
dirty_cafe_df.filter(~(F.regexp_like(F.col("Transaction ID"), F.lit(r"^TXN_(\d{7})$")))).select("Transaction ID").show()

### Transaction ID - Data Cleaning

In [0]:
dirty_cafe_df = dirty_cafe_df.withColumnRenamed("Transaction ID","transaction_id")\
    .withColumn("transaction_id", F.trim(F.regexp_replace(F.col("transaction_id"), r"^TXN_", "")).try_cast("int"))

dirty_cafe_df.show()

### Data Quality Check - Item

In [0]:
dirty_cafe_df.select("Item").distinct().show()

### Item - Data Cleaning

In [0]:
dirty_cafe_df = dirty_cafe_df.withColumnRenamed("Item","item")\
                .withColumn("item", F.lower(F.trim(F.col("item"))))\
                .withColumn("item", F.when(F.col("item").rlike(r"ERROR|UNKNOWN|NULL|error|unknown|null"), F.lit(None)).otherwise(F.col("item")))

dirty_cafe_df.select("item").distinct().show()

### Data Quality Check - Quantity

In [0]:
dirty_cafe_df.select("Quantity").distinct().show()

### Data Cleaning - Quantity

In [0]:
dirty_cafe_df = dirty_cafe_df.withColumnRenamed("Quantity","quantity")\
                .withColumn("quantity",F.trim(F.col("quantity")))\
                .withColumn("quantity",F.col("quantity").try_cast("int"))

In [0]:
dirty_cafe_df.select("quantity").distinct().collect()

### Data Quality Check - Price Per Unit

In [0]:
dirty_cafe_df.select("Price Per Unit").distinct().show()

In [0]:
dirty_cafe_df = dirty_cafe_df.withColumnRenamed("Price Per Unit", "price_per_unit")\
                .withColumn("price_per_unit", F.trim(F.col("price_per_unit")))\
                .withColumn("price_per_unit", F.col("price_per_unit").try_cast("float"))

In [0]:
dirty_cafe_df.select("price_per_unit").distinct().show()

### Data Cleaning - Total Spent

In [0]:
dirty_cafe_df = dirty_cafe_df.withColumnRenamed("Total Spent", "total_spent")\
                .withColumn("total_spent", F.trim(F.col("total_spent")).try_cast("double"))\
                .withColumn("total_spent", F.col("quantity") * F.col("price_per_unit"))

In [0]:
dirty_cafe_df.select("total_spent").distinct().show()

### Data Quality Check - Payment Method

In [0]:
dirty_cafe_df.select("Payment Method").distinct().show()

### Data Cleaning - Payment Method

In [0]:
dirty_cafe_df = dirty_cafe_df.withColumnRenamed("Payment Method", "payment_method")\
                .withColumn("payment_method", F.lower(F.trim(F.col("payment_method"))))\
                .withColumn("payment_method", F.when(F.col("payment_method").rlike(r"ERROR|UNKNOWN|NULL|null|error|unknown"), F.lit(None)).otherwise(F.col("payment_method")))

In [0]:
dirty_cafe_df.show()

### Data Quality Check - Location

In [0]:
dirty_cafe_df.select("Location").distinct().show()

### Data Cleaning - Location

In [0]:
dirty_cafe_df = dirty_cafe_df.withColumnRenamed("Location", "location")\
                .withColumn("location", F.lower(F.trim(F.col("location"))))\
                .withColumn("location", F.when(F.col("location").rlike(r"ERROR|UNKNOWN|NULL|null|error|unknown"), F.lit(None)).otherwise(F.col("location")))

In [0]:
dirty_cafe_df.select("location").distinct().show()

### Data Cleaning - Date

In [0]:
dirty_cafe_df = dirty_cafe_df.withColumnRenamed("Transaction Date", "transaction_date")\
                .withColumn("transaction_date", F.trim(F.col("transaction_date")))\
                .withColumn("transaction_date", F.col("transaction_date").try_cast("date"))

In [0]:
dirty_cafe_df.orderBy(F.col("transaction_date").asc_nulls_first()).show()

### Extract dayofmonth, month, and year to create 3 new columns for easier analytics.

In [0]:
dirty_cafe_df = dirty_cafe_df.withColumn("year", F.year(F.col("transaction_date")))\
                            .withColumn("month", F.month(F.col("transaction_date")))\
                            .withColumn("day", F.dayofmonth(F.col("transaction_date")))

In [0]:
dirty_cafe_df.show()

## Data Cleaning Complete!!

### Load the data to the table.

In [0]:
dirty_cafe_df.write\
    .mode("overwrite")\
    .format("delta")\
    .option("mergeSchema","true")\
    .saveAsTable("de_session_exercises.default.clean_cafe")

### Write the cleaned data to a CSV file.

In [0]:
# Convert small clean dataset to a pandas dataframe 
pandas_df = dirty_cafe_df.toPandas()

# Use local Python execution to write to the Workspace directory safely
pandas_df.to_csv(
    "/Workspace/Users/raj.s@brilworks.com/data-engineering-learnings/exercises/session-4/clean_cafe_sales.csv", 
    index=False
)
